# 03 — Five-Class LSTM for Cyberbullying Text Classification

## Objective

Train a Keras LSTM for the focused five-class task: religion, age, ethnicity, gender, and non-bullying content. The sequence pipeline keeps stopwords and does not lemmatize; the model learns embeddings and processes word order.

This notebook uses the same canonical source filtering and 70/15/15 split as Notebooks 01 and 02. It filters `other_cyberbullying` only after splitting and does **not** drop long tweets: sequences are post-truncated to a fixed length so the five-class test rows remain aligned with the common split.

## Notebook roadmap

1. Load data and confirm shared split fingerprint
2. Select the five-class task after splitting
3. Apply sequence-friendly preprocessing
4. Tokenize and pad sequences
5. Oversample training sequences only
6. Train masked Embedding → LSTM → Dense model
7. Evaluate, visualize training curves, and inspect errors
8. Optionally run a matched six-class LSTM control (extra runtime)

## Environment and data

Install `requirements.txt` from the repository root. Put `cyberbullying_tweets.csv` in `data/` or `/content/` in Colab. A recent TensorFlow/Keras build is recommended because the LSTM uses `use_cudnn=False` to avoid the mask/cuDNN runtime assertion encountered in the earlier notebook.

In [ ]:
from pathlib import Path
import hashlib
import random
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

# `contractions` is listed in requirements.txt. Keep a small fallback so the
# notebook remains usable if a fresh environment has not installed it yet.
try:
    from contractions import fix as fix_contractions
except ImportError:
    def fix_contractions(text):
        text = str(text)
        replacements = [
            (r"\bcan't\b", "cannot"), (r"\bwon't\b", "will not"),
            (r"\bshan't\b", "shall not"), (r"\bain't\b", "is not"),
            (r"n['’]t\b", " not"), (r"['’]re\b", " are"),
            (r"['’]ve\b", " have"), (r"['’]ll\b", " will"),
            (r"['’]d\b", " would"), (r"['’]m\b", " am"),
        ]
        for pattern, replacement in replacements:
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        return text

SEED = 2042
ALL_LABELS = [
    "religion", "age", "ethnicity", "gender",
    "not_cyberbullying", "other_cyberbullying"
]
RETAINED_LABELS = [
    "religion", "age", "ethnicity", "gender", "not_cyberbullying"
]
OTHER_LABEL = "other_cyberbullying"

random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 140)

print("Random seed:", SEED)

## 1. Load data and reproduce the shared split

The next cell is identical to Notebooks 01 and 02. The split fingerprint should match exactly for the same dataset file.

In [ ]:
# Locate the dataset whether this notebook is run from the repository root,
# the notebooks/ folder, or Google Colab after uploading the CSV.
CANDIDATE_PATHS = [
    Path("data/cyberbullying_tweets.csv"),
    Path("../data/cyberbullying_tweets.csv"),
    Path("cyberbullying_tweets.csv"),
    Path("/content/cyberbullying_tweets.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find cyberbullying_tweets.csv. Place it at data/cyberbullying_tweets.csv "
        "or upload it to /content/cyberbullying_tweets.csv in Colab."
    )

raw = pd.read_csv(DATA_PATH).rename(columns={
    "tweet_text": "text",
    "cyberbullying_type": "label",
})
assert {"text", "label"}.issubset(raw.columns), (
    "Expected columns text/label or tweet_text/cyberbullying_type."
)

raw = raw.dropna(subset=["text", "label"]).copy()
raw["text"] = raw["text"].astype(str).str.strip()
raw["label"] = raw["label"].astype(str).str.strip().str.lower()
raw["label"] = raw["label"].replace({
    "not bullying": "not_cyberbullying",
    "not cyberbullying": "not_cyberbullying",
    "other bullying": "other_cyberbullying",
})
raw = raw[raw["text"].ne("") & raw["label"].isin(ALL_LABELS)].copy()

print(f"Dataset path: {DATA_PATH}")
print(f"Rows after basic missing/label checks: {len(raw):,}")
print(f"Raw exact duplicate rows: {raw.duplicated(subset=['text', 'label']).sum():,}")

# Conservative, shared deduplication key. This is deliberately separate from
# either model's actual text-cleaning function. It reduces the chance that
# punctuation/URL/mention variants of the same tweet cross data partitions.
def make_split_key(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"@\w+", " MENTIONTOKEN ", text)
    text = re.sub(r"#([\w-]+)", r"\1", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split()).strip()

raw["split_key"] = raw["text"].map(make_split_key)
raw = raw[raw["split_key"].ne("")].copy()

# If the same normalized key carries more than one original label, preserve
# those rows for inspection but exclude that key from model evaluation.
label_count_per_key = raw.groupby("split_key")["label"].nunique()
conflicting_keys = set(label_count_per_key[label_count_per_key > 1].index)
label_conflicts = raw[raw["split_key"].isin(conflicting_keys)].copy()

model_source = raw[~raw["split_key"].isin(conflicting_keys)].copy()
rows_before_dedup = len(model_source)
model_df = model_source.drop_duplicates(subset="split_key", keep="first").copy()
model_df = model_df.reset_index(drop=True)
model_df["row_id"] = np.arange(len(model_df), dtype=np.int64)

print(f"Normalized text keys with conflicting labels: {len(conflicting_keys):,}")
print(f"Rows held aside due to conflicting labels: {len(label_conflicts):,}")
print(f"Repeated normalized keys removed: {rows_before_dedup - len(model_df):,}")
print(f"Rows available for modelling: {len(model_df):,}")

if not label_conflicts.empty:
    print("Examples of conflicting normalized keys (inspect rather than assume annotation error):")
    display(label_conflicts[["text", "label", "split_key"]].head(10))

observed_labels = set(model_df["label"].unique())
assert observed_labels == set(ALL_LABELS), (
    f"Expected all six labels {ALL_LABELS}, found {sorted(observed_labels)}"
)

# One shared, deterministic 70/15/15 split from the ORIGINAL six-class data.
# Filter out 'other_cyberbullying' only after these partitions are created.
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=SEED,
    stratify=model_df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

split_parts = {"train": train_df, "validation": val_df, "test": test_df}
print(f"Train: {len(train_df):,} ({len(train_df)/len(model_df):.1%})")
print(f"Validation: {len(val_df):,} ({len(val_df)/len(model_df):.1%})")
print(f"Test: {len(test_df):,} ({len(test_df)/len(model_df):.1%})")

split_counts = pd.DataFrame({
    name: part["label"].value_counts()
    for name, part in split_parts.items()
}).fillna(0).astype(int).reindex(ALL_LABELS)
display(split_counts)

# This fingerprint should match across all three notebooks when using the same CSV.
fp_payload = "|".join(
    f"{name}:{','.join(map(str, sorted(part['row_id'].tolist())))}"
    for name, part in split_parts.items()
)
split_fingerprint = hashlib.sha256(fp_payload.encode("utf-8")).hexdigest()[:12]
print("Shared split fingerprint:", split_fingerprint)
print("Record this fingerprint when comparing results from the three notebooks.")

## 2. Filter to the five-class scope

Filter after creating the original six-class split. The original test partition is preserved, and we only remove `other_cyberbullying` from each partition for this specific model.

In [ ]:
train_5 = train_df[train_df["label"].isin(RETAINED_LABELS)].copy()
val_5 = val_df[val_df["label"].isin(RETAINED_LABELS)].copy()
test_5 = test_df[test_df["label"].isin(RETAINED_LABELS)].copy()

assert set(train_5["label"].unique()) == set(RETAINED_LABELS)
assert set(val_5["label"].unique()) == set(RETAINED_LABELS)
assert set(test_5["label"].unique()) == set(RETAINED_LABELS)

print("Train rows:", len(train_5))
print("Validation rows:", len(val_5))
print("Test rows:", len(test_5))
print("Longest original cleaned-text length will be handled by truncation, not row deletion.")
display(pd.DataFrame({
    "train": train_5["label"].value_counts().reindex(RETAINED_LABELS),
    "validation": val_5["label"].value_counts().reindex(RETAINED_LABELS),
    "test": test_5["label"].value_counts().reindex(RETAINED_LABELS),
}).fillna(0).astype(int))

## 3. Sequence-friendly preprocessing

Compared with the classical notebook, this function keeps stopwords and skips lemmatization. It still expands contractions, normalizes mentions/URLs/hashtags, lowercases, removes punctuation and digits, and normalizes whitespace. This is a modelling choice to test—not a guarantee that lighter preprocessing always improves performance.

In [ ]:
import tensorflow as tf
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping


def clean_tweet_sequence(text):
    """Normalize tweet text while keeping stopwords and word forms."""
    text = "" if pd.isna(text) else str(text)
    text = fix_contractions(text)
    text = re.sub(r"\r|\n", " ", text.lower())
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"@\w+", " MENTIONTOKEN ", text)
    text = re.sub(r"#([\w-]+)", r"\1", text)
    text = re.sub(r"[^\x00-\x7F]", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = " ".join(text.split()).strip()
    return text if text else "emptytext"

print("TensorFlow version:", tf.__version__)
print("Sequence preprocessing ready; stopwords and word forms are retained.")

In [ ]:
X_train_text_5 = train_5["text"].map(clean_tweet_sequence).tolist()
X_val_text_5 = val_5["text"].map(clean_tweet_sequence).tolist()
X_test_text_5 = test_5["text"].map(clean_tweet_sequence).tolist()
y_train_5 = train_5["label"].to_numpy()
y_val_5 = val_5["label"].to_numpy()
y_test_5 = test_5["label"].to_numpy()

text_lengths = pd.Series(X_train_text_5).str.split().str.len()
print(text_lengths.describe().round(2))
plt.figure(figsize=(9, 4))
sns.histplot(text_lengths, bins=40)
plt.title("Training tweet token counts after sequence preprocessing")
plt.xlabel("Tokens")
plt.tight_layout()
plt.show()

## 4. Tokenization, vocabulary, and right-padding

The tokenizer is fitted only on the five-class training text. Validation and test texts do not contribute to the vocabulary. Sequences are post-padded and post-truncated to 60 tokens; this avoids dropping rows from the common test set.

In [ ]:
VOCAB_LIMIT = 20000
MAX_SEQUENCE_LENGTH = 60
OOV_TOKEN = "<OOV>"

tokenizer = Tokenizer(num_words=VOCAB_LIMIT, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train_text_5)
vocab_size_actual = min(VOCAB_LIMIT, len(tokenizer.word_index) + 1)


def to_padded(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(
        sequences,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post",
    )

X_train_seq = to_padded(X_train_text_5)
X_val_seq = to_padded(X_val_text_5)
X_test_seq = to_padded(X_test_text_5)

print("Tokenizer word_index size:", len(tokenizer.word_index))
print("Embedding input dimension:", vocab_size_actual)
print("Train/validation/test shapes:", X_train_seq.shape, X_val_seq.shape, X_test_seq.shape)

# cuDNN-backed recurrent layers require strictly right-padded inputs when masked.
def assert_right_padded(arr, name):
    zero_seen = np.maximum.accumulate(arr == 0, axis=1)
    assert not np.any((arr != 0) & zero_seen), f"Non-right-padded sequence found in {name}"
    print(f"{name}: sequences are right-padded")

assert_right_padded(X_train_seq, "train")
assert_right_padded(X_val_seq, "validation")
assert_right_padded(X_test_seq, "test")

# Print one transparent preprocessing example.
print("\nRaw:", train_5["text"].iloc[0])
print("Clean:", X_train_text_5[0])
print("Token ids:", tokenizer.texts_to_sequences([X_train_text_5[0]])[0][:25])
print("Padded ids:", X_train_seq[0][:25])

## 5. Balance only the training sequences

The tokenizer is fitted **before** oversampling, so duplicate oversampled records do not influence vocabulary construction. Oversampling is then applied to the fixed-length training sequences; validation and test data stay untouched.

In [ ]:
label_to_id = {label: i for i, label in enumerate(RETAINED_LABELS)}
id_to_label = {i: label for label, i in label_to_id.items()}
y_train_ids = np.array([label_to_id[v] for v in y_train_5], dtype=np.int64)
y_val_ids = np.array([label_to_id[v] for v in y_val_5], dtype=np.int64)
y_test_ids = np.array([label_to_id[v] for v in y_test_5], dtype=np.int64)

ros = RandomOverSampler(random_state=SEED)
X_train_seq_bal, y_train_ids_bal = ros.fit_resample(X_train_seq, y_train_ids)

print("Before oversampling:", X_train_seq.shape, pd.Series(y_train_ids).value_counts().sort_index().to_dict())
print("After oversampling:", X_train_seq_bal.shape, pd.Series(y_train_ids_bal).value_counts().sort_index().to_dict())

## 6. Build the LSTM

Architecture: **Embedding → LSTM → Dropout → Dense → Softmax**.

`mask_zero=True` tells the recurrent layer to ignore padding ID 0. `use_cudnn=False` avoids the mask/cuDNN assertion observed in the earlier environment; it may run more slowly than the accelerated kernel.

In [ ]:
tf.keras.utils.set_random_seed(SEED)

model = models.Sequential([
    layers.Input(shape=(MAX_SEQUENCE_LENGTH,)),
    layers.Embedding(
        input_dim=vocab_size_actual,
        output_dim=128,
        mask_zero=True,
        name="word_embedding",
    ),
    layers.LSTM(128, use_cudnn=False, name="lstm"),
    layers.Dropout(0.35),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(len(RETAINED_LABELS), activation="softmax", name="classifier"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 7. Train with early stopping

Validation macro-F1 is not used for early stopping here; validation loss is monitored and the best validation-loss weights are restored. The test set remains untouched until final evaluation.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    X_train_seq_bal,
    y_train_ids_bal,
    validation_data=(X_val_seq, y_val_ids),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=2,
)

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_seq, y_test_ids, verbose=0)
y_prob = model.predict(X_test_seq, verbose=0)
y_pred_ids = np.argmax(y_prob, axis=1)
y_pred_5 = np.array([id_to_label[i] for i in y_pred_ids])

lstm_accuracy_5 = accuracy_score(y_test_5, y_pred_5)
lstm_macro_f1_5 = f1_score(y_test_5, y_pred_5, labels=RETAINED_LABELS, average="macro")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {lstm_accuracy_5:.4f}")
print(f"Test macro-F1: {lstm_macro_f1_5:.4f}")
print(classification_report(
    y_test_5, y_pred_5,
    labels=RETAINED_LABELS,
    target_names=RETAINED_LABELS,
    digits=3,
    zero_division=0,
))

In [ ]:
cm_lstm_5 = confusion_matrix(y_test_5, y_pred_5, labels=RETAINED_LABELS)
cm_lstm_5_norm = confusion_matrix(
    y_test_5, y_pred_5, labels=RETAINED_LABELS, normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(cm_lstm_5, annot=True, fmt="d", cmap="Blues",
            xticklabels=RETAINED_LABELS, yticklabels=RETAINED_LABELS, ax=axes[0])
axes[0].set_title("LSTM — counts")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
sns.heatmap(cm_lstm_5_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=RETAINED_LABELS, yticklabels=RETAINED_LABELS, ax=axes[1])
axes[1].set_title("LSTM — row-normalized")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy by epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Error analysis

The largest confusion pair tells us which categories are hardest for this model to separate. The examples are useful for qualitative inspection, but a prediction error does not by itself prove the source label is wrong.

In [ ]:
cm_no_diag = cm_lstm_5.copy()
np.fill_diagonal(cm_no_diag, 0)
worst_i, worst_j = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)
print(f"Largest confusion: true={RETAINED_LABELS[worst_i]} -> predicted={RETAINED_LABELS[worst_j]} "
      f"({cm_lstm_5[worst_i, worst_j]} test examples)")

error_view = pd.DataFrame({
    "raw_text": test_5["text"].to_numpy(),
    "clean_text": X_test_text_5,
    "true_label": y_test_5,
    "predicted_label": y_pred_5,
    "confidence": y_prob.max(axis=1),
})
display(error_view[
    (error_view["true_label"] == RETAINED_LABELS[worst_i]) &
    (error_view["predicted_label"] == RETAINED_LABELS[worst_j])
].sort_values("confidence", ascending=False).head(15))

## 9. Optional matched six-class LSTM control

This trains a second LSTM using the same shared split and preprocessing, but all six labels. It is intentionally disabled by default because training a second recurrent model approximately doubles model-training time. Enable it if you want a controlled five-vs-six-class LSTM comparison. The retained-class test rows are the same rows used by the five-class model above.

In [ ]:
RUN_SIX_CLASS_LSTM_CONTROL = False

if RUN_SIX_CLASS_LSTM_CONTROL:
    # Fit a separate six-class tokenizer on the original six-class training texts only.
    X_train_text_6 = train_df["text"].map(clean_tweet_sequence).tolist()
    X_val_text_6 = val_df["text"].map(clean_tweet_sequence).tolist()
    X_test_text_6 = test_df["text"].map(clean_tweet_sequence).tolist()
    y_train_ids_6 = np.array([ALL_LABELS.index(v) for v in train_df["label"]], dtype=np.int64)
    y_val_ids_6 = np.array([ALL_LABELS.index(v) for v in val_df["label"]], dtype=np.int64)
    y_test_ids_6 = np.array([ALL_LABELS.index(v) for v in test_df["label"]], dtype=np.int64)

    tokenizer_6 = Tokenizer(num_words=VOCAB_LIMIT, oov_token=OOV_TOKEN)
    tokenizer_6.fit_on_texts(X_train_text_6)
    vocab_size_6 = min(VOCAB_LIMIT, len(tokenizer_6.word_index) + 1)

    def to_padded_6(texts):
        return pad_sequences(
            tokenizer_6.texts_to_sequences(texts),
            maxlen=MAX_SEQUENCE_LENGTH,
            padding="post",
            truncating="post",
        )

    X_train_seq_6 = to_padded_6(X_train_text_6)
    X_val_seq_6 = to_padded_6(X_val_text_6)
    X_test_seq_6 = to_padded_6(X_test_text_6)
    X_train_seq_6_bal, y_train_ids_6_bal = RandomOverSampler(
        random_state=SEED
    ).fit_resample(X_train_seq_6, y_train_ids_6)

    tf.keras.utils.set_random_seed(SEED)
    model_6 = models.Sequential([
        layers.Input(shape=(MAX_SEQUENCE_LENGTH,)),
        layers.Embedding(vocab_size_6, 128, mask_zero=True, name="word_embedding_6"),
        layers.LSTM(128, use_cudnn=False, name="lstm_6"),
        layers.Dropout(0.35),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(len(ALL_LABELS), activation="softmax", name="classifier_6"),
    ])
    model_6.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history_6 = model_6.fit(
        X_train_seq_6_bal, y_train_ids_6_bal,
        validation_data=(X_val_seq_6, y_val_ids_6),
        epochs=15, batch_size=64,
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        verbose=2,
    )

    pred_ids_6 = np.argmax(model_6.predict(X_test_seq_6, verbose=0), axis=1)
    y_pred_6 = np.array([ALL_LABELS[i] for i in pred_ids_6])
    retained_mask_6 = test_df["label"].isin(RETAINED_LABELS).to_numpy()
    y_true_common = test_df.loc[retained_mask_6, "label"].to_numpy()
    y_pred_6_common = y_pred_6[retained_mask_6]

    comparison = pd.DataFrame([
        {
            "formulation": "Six-class LSTM scored on retained-class test rows",
            "accuracy": accuracy_score(y_true_common, y_pred_6_common),
            "macro_f1_over_retained_classes": f1_score(
                y_true_common, y_pred_6_common,
                labels=RETAINED_LABELS, average="macro", zero_division=0
            ),
            "predicted_other_on_retained_rows": int(np.sum(y_pred_6_common == OTHER_LABEL)),
        },
        {
            "formulation": "Five-class LSTM",
            "accuracy": accuracy_score(y_test_5, y_pred_5),
            "macro_f1_over_retained_classes": f1_score(
                y_test_5, y_pred_5,
                labels=RETAINED_LABELS, average="macro", zero_division=0
            ),
            "predicted_other_on_retained_rows": 0,
        },
    ])
    display(comparison.round(4))
else:
    print("Skipped. Set RUN_SIX_CLASS_LSTM_CONTROL = True to train the matched six-class control.")

## 10. Final interpretation

Report the final test accuracy, macro-F1, per-class metrics, confusion matrix, and error examples. Do not compare with the old six-class LSTM score as though it were controlled unless you run the optional matched six-class control above.

The model predicts dataset-defined categories. It does not determine a post's legal status, its severity, or which moderation action should be taken.